# BattMo.jl Workshop — Common Workflows

**Agenda slot:** 11:00 – 12:00 (Hands-on session 1)

In this session we'll walk through workflows that go a bit further than the Hello World basics from this morning. You will:
- Learn how to create your own cell parameter sets from a model template
- Simulate a WLTP drive cycle to learn how to define your own parameter functions
- Calibrate model parameters against experimental voltage data
- Run a thermal simulation

These same skills (modifying parameters, defining custom functions, fitting parameters to data) are exactly what you'll need for this afternoon's case-solving session.

In [92]:
using BattMo, GLMakie, Jutul, CSV, DataFrames

## 1 - Create your own parameter sets

Cell parameter sets define the physical and chemical properties of the battery system you're simulating. You can build them from scratch using your model setup, modify them, and save them for future use.

The difficulty with setting up your own input parameters is that it's often not very clear which parameters you need for which submodel combination. BattMo provides functions that create empty parameter sets containing exactly the required keys for your chosen model setup, so you don't need to guess.

### Step 1 - Initiate a model setup

Model settings specify which submodels you'd like included in the simulation. Let's have a quick look at the submodels you can configure.

In [93]:
print_submodels()

ℹ️  Submodels Information
Parameter                     Options                                           Documentation
----------------------------------------------------------------------------------------------------
ButlerVolmer                  Standard, Chayambuka                              [visit](https://battmoteam.github.io/BattMo.jl/dev/manuals/user_guide/pxd_model)

CurrentCollectors             Standard                                          -         
SEIModel                      Bolay                                             [visit](https://battmoteam.github.io/BattMo.jl/dev/manuals/user_guide/sei_model)

RampUp                        Sinusoidal                                        [visit](https://battmoteam.github.io/BattMo.jl/dev/manuals/user_guide/ramp_up)

TemperatureDependence         Arrhenius                                         [visit](https://battmoteam.github.io/BattMo.jl/dev/manuals/user_guide/arrhenius)

PotentialFlowDiscretization   GeneralAD, T

Let's set up our own cell parameter set for a P2D simulation that includes SEI growth. We start from the default P2D model settings.

In [94]:
model_settings = load_model_settings(; from_default_set = "p2d")

print_info(model_settings)


PARAMETER OVERVIEW
Parameter                                                                                 Unit                Type                Value                         
----------------------------------------------------------------------------------------------------------------------------------------------------------------
[ "ButlerVolmer" ]                                                                        N/A                 String              Standard                      
[ "ModelFramework" ]                                                                      N/A                 String              P2D                           
[ "PotentialFlowDiscretization" ]                                                         N/A                 String              GeneralAD                     
[ "RampUp" ]                                                                              N/A                 String              Sinusoidal                    
[ "TransportIn

These settings don't include the SEI model yet, so let's add it.

In [95]:
model_settings["SEIModel"] = "Bolay"

print_info(model_settings)


PARAMETER OVERVIEW
Parameter                                                                                 Unit                Type                Value                         
----------------------------------------------------------------------------------------------------------------------------------------------------------------
[ "ButlerVolmer" ]                                                                        N/A                 String              Standard                      
[ "ModelFramework" ]                                                                      N/A                 String              P2D                           
[ "PotentialFlowDiscretization" ]                                                         N/A                 String              GeneralAD                     
[ "RampUp" ]                                                                              N/A                 String              Sinusoidal                    
[ "SEIModel" ]

Now we can instantiate the Lithium-Ion Battery model and check whether our model settings are valid.

In [96]:
model_setup = LithiumIonBattery(; model_settings)

✔️ Validation of ModelSettings passed: No issues found.
──────────────────────────────────────────────────


LithiumIonBattery("Setup object for a P2D lithium-ion model", {
    "ButlerVolmer" => "Standard"
    "SEIModel" => "Bolay"
    "RampUp" => "Sinusoidal"
    "Metadata" =>     {
        "Description" => "Default model settings for a P2D simulation including a current ramp up, excluding current collectors and SEI effects."
        "Title" => "p2d"
    }
    "PotentialFlowDiscretization" => "GeneralAD"
    "TransportInSolid" => "FullDiffusion"
    "ModelFramework" => "P2D"
}, true, #undef)

### Step 2 - Load an empty parameter set

Next, we create an empty parameter dictionary based on our model. This includes all the required keys, without any values filled in.

In [97]:
empty_cell_parameter_set = load_cell_parameters(; from_model_template = model_setup)

print_info(empty_cell_parameter_set)


PARAMETER OVERVIEW
Parameter                                                                                 Unit                Type                Value                         
----------------------------------------------------------------------------------------------------------------------------------------------------------------
[ "Cell" ][ "ElectrodeGeometricSurfaceArea" ]                                             m²                  Float64             0.0                           
[ "Electrolyte" ][ "ChargeNumber" ]                                                       1                   Float64             0.0                           
[ "Electrolyte" ][ "Concentration" ]                                                      mol·m⁻³             Float64             0.0                           
[ "Electrolyte" ][ "Density" ]                                                            kg·m⁻³              Float64             0.0                           
[ "Electrolyte

Let's check that it indeed contains the SEI input parameters — these are part of the `Interphase` parameters.

In [98]:
empty_cell_parameter_set["NegativeElectrode"]["Interphase"]

Dict{Any, Any} with 7 entries:
  "ElectronicDiffusionCoefficient" => 0.0
  "InterstitialConcentration"      => 0.0
  "InitialThickness"               => 0.0
  "IonicConductivity"              => 0.0
  "StoichiometricCoefficient"      => 0.0
  "InitialPotentialDrop"           => 0.0
  "MolarVolume"                    => 0.0

### Step 3 - Save the empty set to a JSON file

We can now write this empty set to a JSON file so we can fill it in (in a text editor, or via the dictionary interface) and reuse it later.

In [99]:
file_path = "my_custom_parameters.json"
write_to_json_file(file_path, empty_cell_parameter_set)

Data successfully written to my_custom_parameters.json


### Step 4 - Let BattMo help you fill the empty set

If you're unsure what a specific parameter means or how it should be formatted, BattMo provides a helpful function to inspect any parameter.

In [100]:
print_info("OpenCircuitPotential", view = "CellParameters")


----------------------------------------------------------------------------------------------------
🔋  Cell Parameter:  OpenCircuitPotential
----------------------------------------------------------------------------------------------------
    🔹 Name               OpenCircuitPotential
    🔹 Category           CellParameters
    🔹 Description        The open-circuit potential of the active material under a given intercalant stoichimetry and temperature.
    🔹 Type               String, Dict{String, Vector}, Real
    🔹 Unit               V
    🔹 Documentation      [visit](https://battmo.org/BattMo.jl/dev/manuals/user_guide/simulation_dependent_input)

    🔹 Ontology link      [visit](https://w3id.org/emmo/domain/electrochemistry#electrochemistry_9c657fdc_b9d3_4964_907c_f9a6e8c5f52b)




In [101]:
print_info("concentration")


----------------------------------------------------------------------------------------------------
🔋  Cell Parameter:  InterstitialConcentration
----------------------------------------------------------------------------------------------------
    🔹 Name               InterstitialConcentration
    🔹 Category           CellParameters
    🔹 Description        Amount of neutral intersticial sites of the intercalant (Li+) per volume of solid. See https://doi.org/10.1002/cssc.201800077.
    🔹 Type               Real
    🔹 Unit               mol·m⁻³
    🔹 Minimum value      0.0
    🔹 Maximum value      100000.0

----------------------------------------------------------------------------------------------------
🔋  Cell Parameter:  MaximumConcentration
----------------------------------------------------------------------------------------------------
    🔹 Name               MaximumConcentration
    🔹 Category           CellParameters
    🔹 Description        The saturation concentratio

The automatic validation when initiating the `Simulation` object will help you keep track of your progress and flag any unrealistic values.

In [102]:
# --- fill in my_custom_parameters.json, then come back and run this cell ---

cell_parameters = load_cell_parameters(; from_file_path = "my_custom_parameters.json")

sim = Simulation(model_setup, cell_parameters, load_cycling_protocol(; from_default_set = "cc_discharge"))

Excessive output truncated after 524363 bytes.

🔍 Validation of CellParameters failed with 35 issues:

──────────────────────────────────────────────────
──────────────────────────────────────────────────
Issue 1:
📍 Where:       [Electrolyte][TransferenceNumber]
🔢 Provided:    0.0
🔑 Rule:        minimum = 0.01
🛠  Issue:       Value is below the minimum allowed (0.01)

──────────────────────────────────────────────────
Issue 2:
📍 Where:       [Electrolyte][Density]
🔢 Provided:    0.0
🔑 Rule:        minimum = 0.01
🛠  Issue:       Value is below the minimum allowed (0.01)

──────────────────────────────────────────────────
Issue 3:
📍 Where:       [Electrolyte][ChargeNumber]
🔢 Provided:    0.0
🔑 Rule:        minimum = 1
🛠  Issue:       Value is below the minimum allowed (1)

──────────────────────────────────────────────────
Issue 4:
📍 Where:       [Cell][ElectrodeGeometricSurfaceArea]
🔢 Provided:    0.0
🔑 Rule:        minimum = 0.0001
🛠  Issue:       Value is below the minimum allowed (0.0001)

─────────────────────────────────────────

Simulation(LithiumIonBattery("Setup object for a P2D lithium-ion model", {
    "ButlerVolmer" => "Standard"
    "SEIModel" => "Bolay"
    "RampUp" => "Sinusoidal"
    "Metadata" =>     {
        "Description" => "Default model settings for a P2D simulation including a current ramp up, excluding current collectors and SEI effects."
        "Title" => "p2d"
    }
    "PotentialFlowDiscretization" => "GeneralAD"
    "TransportInSolid" => "FullDiffusion"
    "ModelFramework" => "P2D"
}, true, MultiModel{:IntercalationBattery, JutulStorage{Nothing}, Vector{Jutul.CrossTermPair}, Nothing, DefaultContext, Dict{Symbol, Int64}}(JutulStorage{Nothing}(OrderedCollections.OrderedDict{Symbol, Any}(:NegativeElectrodeActiveMaterial => SimulationModel{DiscretizedDomain{DataDomain{UnstructuredMesh{3, Nothing, Nothing, Vector{Int64}, Jutul.IndirectionMap{Int64}, Float64, Nothing, Nothing, Int64}, Dict{JutulEntity, Int64}, OrderedCollections.OrderedDict{Symbol, Any}}, @NamedTuple{flow::PotentialFlow{:gener

>Now you know how to create your own input files from scratch!

## 2 - Create your own parameter function

For some parameters you can define your own function in a Julia script and pass it to BattMo. This is available for:

**Cell parameters**
- OpenCircuitPotential (negative and positive electrode)
- DiffusionCoefficient (negative and positive electrode)
- ReactionRateConstant (negative and positive electrode)
- DiffusionCoefficient (electrolyte)
- Conductivity (electrolyte)

**Cycling protocol parameters**
- Current

Let's have a look at how this works.

For the cell parameters, let's look at the `xu_2015` default set and see how its open circuit potential is defined.

In [103]:
cell_parameters = load_cell_parameters(; from_default_set = "xu_2015")

cell_parameters["NegativeElectrode"]["ActiveMaterial"]["OpenCircuitPotential"]

Dict{String, Any} with 2 entries:
  "FilePath"     => "function_parameters_Xu2015.jl"
  "FunctionName" => "open_circuit_potential_graphite_Xu_2015"

We can see that the `OpenCircuitPotential` here is a `Dict` containing:
- **FilePath**: the path to the Julia script containing the function (relative to the JSON file)
- **FunctionName**: the name of the function

When running a simulation, BattMo will access that script and import the function into its namespace so it can be used internally.

An easy way to check if your functional parameters are defined correctly is to plot them all at once.

In [104]:
plot_cell_curves(cell_parameters)

In [105]:
GLMakie.closeall()

An even easier way to use a functional parameter is to define a function anywhere and import it into `Main` — the namespace BattMo can access during a simulation. You can define it directly in your script, or in a separate file included via `include("my_functions.jl")`.

Let's go through a fun example: setting up your own current function from drive-cycle data.

### Current function

We'll use Worldwide harmonized Light vehicles Test Procedure (WLTP) drive cycle data stored in `data/wltp.csv`. To create a current function from this data:
1. Read the time and power data from the CSV.
2. Create an interpolation object from the data.
3. Define a function that returns the current, calculated from the interpolated object.

In [106]:
# An example of a user defined current function using WLTP data from https://github.com/JRCSTU/wltp

# 1. Read data
path = joinpath(@__DIR__, "data", "wltp.csv")
df = CSV.read(path, DataFrame)

t = df[:, 1]
P = df[:, 2]

# 2. Use a Jutul interpolator to create an interpolation object of the time and power data
power_func = Jutul.get_1d_interpolator(t, P, cap_endpoints = false)

# 3. Define a function to calculate the current. The current function has to accept time and voltage as arguments.
function wltp_current(time, voltage)

    factor = 4000 # Account for the fact that we're simulating a single cell instead of a battery pack

    return power_func(time) / voltage / factor
end

wltp_current (generic function with 1 method)

Now that our function is defined in `Main`, we need a cycling protocol that tells BattMo to use it. Cycling protocols are categorized by the `Protocol` parameter — let's see what values it can take.

In [107]:
print_info("Protocol")

`Protocol` can be `CC`, `CCCV`, or `Function`. We want our own current function, so we use `Function`. Let's load the default `user_defined_current_function` cycling protocol and adapt it.

In [108]:
current_function = load_cycling_protocol(; from_default_set = "user_defined_current_function")

print_info(current_function)

We need to change the function name to ours — the other parameters can stay as they are for now.

In [109]:
current_function["FunctionName"] = "wltp_current"
print_info(current_function)

Now we can run a simulation using this cycling protocol.

In [110]:
cell_parameters = load_cell_parameters(; from_default_set = "chen_2020")

model_setup = LithiumIonBattery()

sim = Simulation(model_setup, cell_parameters, current_function)

output = solve(sim)

plot_dashboard(output)

The output curves don't look very smooth — our time resolution probably isn't fine enough. Let's check which simulation setting controls this.

In [111]:
print_info("time", view = "SimulationSettings")

`TimeStepDuration` is what we need. So far we've only used the default simulation settings — let's check its default value.

In [112]:
sim.settings["TimeStepDuration"]

50

It's 50 seconds. Let's refine the time resolution to 1 second. First, load the default simulation settings.

In [113]:
simulation_settings = load_simulation_settings(; from_default_set = "p2d")
simulation_settings["TimeStepDuration"] = 1

1

Now pass the simulation settings to the `Simulation` object and re-run.

In [114]:
sim = Simulation(model_setup, cell_parameters, current_function; simulation_settings)

output = solve(sim)

plot_dashboard(output)

In [115]:
GLMakie.closeall()

## 3 - Thermal simulations

So far every simulation has been isothermal — no temperature evolution. BattMo.jl has two thermal models: **Decoupled** (the thermal field is solved only after the electrochemical simulation finishes) and **Sequential** (a loosely coupled electro-thermal model that solves electrochemistry and temperature one after the other at each time step).

Thermal simulations have two extra requirements compared to what we've done so far:
1. They need the full 3D **P4D framework** (Pouch or Cylindrical) with current collectors enabled — temperature gradients aren't meaningful in a 1D P2D model.
2. Every cell component needs `SpecificHeatCapacity` and `ThermalConductivity`, and every active material needs `EntropyChange` — the `chen_2020` set we've used so far doesn't include these, but `xu_2015` does, so we'll switch to it for this section.

Let's configure a P4D Pouch model with current collectors and the Sequential thermal model.

In [116]:
model_settings = load_model_settings(; from_default_set = "p2d")
model_settings["ModelFramework"] = "P4D Pouch"
model_settings["CurrentCollectors"] = "Standard"
model_settings["ThermalModel"] = "Sequential"

model = LithiumIonBattery(; model_settings)

LithiumIonBattery("Setup object for a P4D Pouch lithium-ion model", {
    "ButlerVolmer" => "Standard"
    "CurrentCollectors" => "Standard"
    "RampUp" => "Sinusoidal"
    "Metadata" =>     {
        "Description" => "Default model settings for a P2D simulation including a current ramp up, excluding current collectors and SEI effects."
        "Title" => "p2d"
    }
    "PotentialFlowDiscretization" => "GeneralAD"
    "TransportInSolid" => "FullDiffusion"
    "ThermalModel" => "Sequential"
    "ModelFramework" => "P4D Pouch"
}, true, #undef)

Now let's load the `xu_2015` cell parameters and a constant current discharge protocol, and set an initial and ambient temperature (in Kelvin — BattMo works in SI units).

In [117]:
cell_parameters = load_cell_parameters(; from_default_set = "xu_2015")
cycling_protocol = load_cycling_protocol(; from_default_set = "cc_discharge")

cycling_protocol["DRate"] = 1.0
cycling_protocol["InitialTemperature"] = 25 + 273.15
cycling_protocol["AmbientTemperature"] = 25 + 273.15

298.15

Now let's build and solve the simulation. This is a full 3D simulation, so it takes noticeably longer than the P2D runs we've done so far — expect roughly a minute or two.

In [118]:
sim = Simulation(model, cell_parameters, cycling_protocol)
output = solve(sim);

Temperature is a state variable, nested per component just like the other electrochemical states. Let's plot the maximum temperature in each electrode's active material over time.

In [119]:
time = output.time_series["Time"] ./ 3600

T_ne = vec(maximum(output.states["NegativeElectrode"]["ActiveMaterial"]["Temperature"], dims = 2)) .- 273.15
T_pe = vec(maximum(output.states["PositiveElectrode"]["ActiveMaterial"]["Temperature"], dims = 2)) .- 273.15

f = Figure(size = (700, 400))
ax = Axis(f[1, 1], title = "Maximum electrode temperature vs. time", xlabel = "Time / h", ylabel = "Temperature / degC")
scatterlines!(ax, time, T_ne; label = "Negative electrode", linewidth = 4)
scatterlines!(ax, time, T_pe; label = "Positive electrode", linewidth = 4)
axislegend(ax, position = :lt)
f

You can plot the 3D data interactively.

In [120]:
plot_interactive_3d(output; colormap = :curl)

Or plot a certain component in 2D.

In [121]:
fig_phi, ax_phi = plot_cell_data(
    output.states["NegativeElectrode"]["ActiveMaterial"]["Position"],
    output.states["NegativeElectrode"]["ActiveMaterial"]["Temperature"][end, :];
    colormap = :viridis,
)
ax_phi.aspect = :data
ax_phi.title = "Negative active material Temperature"

fig_phi

In [122]:
GLMakie.closeall()

Try changing `AmbientTemperature` to a hot summer day (e.g. 40°C) and re-running — does the cell stabilize at a higher temperature, or does the gap to the negative electrode grow? This is exactly the kind of check you'll be able to apply to your own design this afternoon.

## 4 - Calibration

The parameters we obtain from characterizing a cell often produce simulations that don't quite match real cycling data. Before trusting a model, it's good practice to **calibrate**: fit a small subset of parameters so the simulated voltage curve matches an experimental one.

We'll calibrate the `xu_2015` parameter set against an experimental 0.5C discharge voltage curve. Let's load the experimental data first.

In [123]:
expdata_05C = CSV.read("data/xu_2015_voltage_curve_05C.csv", DataFrame);

Now let's run a baseline simulation with the original (uncalibrated) parameters at the matching discharge rate, and compare it to the experimental data.

In [124]:
cell_parameters_original = load_cell_parameters(; from_default_set = "xu_2015")
cycling_protocol = load_cycling_protocol(; from_default_set = "cc_discharge")

cycling_protocol["LowerVoltageLimit"] = 2.25
cycling_protocol["DRate"] = 0.5

model_setup = LithiumIonBattery()

sim_original = Simulation(model_setup, cell_parameters_original, cycling_protocol)
output_original = solve(sim_original);

In [125]:
simdata_time_original = output_original.time_series["Time"]
simdata_voltage_original = output_original.time_series["Voltage"]

f = Figure(size = (700, 400))
ax = Axis(f[1, 1], title = "0.5C discharge: simulation vs. experiment", xlabel = "Time / s", ylabel = "Voltage / V")
lines!(ax, simdata_time_original, simdata_voltage_original; linewidth = 4, label = "Simulation: original parameters")
scatter!(ax, expdata_05C[:, 1], expdata_05C[:, 2]; markersize = 12, label = "Experimental data")
axislegend(ax, position = :lb)
f

The simulation doesn't match the experiment particularly well. Let's set up a `VoltageCalibration` to fit it. By default **all parameters are frozen** — we need to explicitly free the ones we want to calibrate, with bounds. Here we free the active material stoichiometric coefficients at SOC 0 and SOC 100 for both electrodes — these effectively control how much of each electrode's capacity is actually used.

In [126]:
calibration = VoltageCalibration(expdata_05C[:, 1], expdata_05C[:, 2], sim_original) # VoltageCalibration(experimental_time, experimental_voltage, simulation)

free_calibration_parameter!(calibration,
    ["NegativeElectrode", "ActiveMaterial", "StoichiometricCoefficientAtSOC100"];
    lower_bound = 0.0, upper_bound = 1.0)
free_calibration_parameter!(calibration,
    ["PositiveElectrode", "ActiveMaterial", "StoichiometricCoefficientAtSOC100"];
    lower_bound = 0.0, upper_bound = 1.0)
free_calibration_parameter!(calibration,
    ["NegativeElectrode", "ActiveMaterial", "StoichiometricCoefficientAtSOC0"];
    lower_bound = 0.0, upper_bound = 1.0)
free_calibration_parameter!(calibration,
    ["PositiveElectrode", "ActiveMaterial", "StoichiometricCoefficientAtSOC0"];
    lower_bound = 0.0, upper_bound = 1.0);

We can check the free parameters, their bounds and current values with `print_info`.

In [127]:
print_info(calibration)

Solving the calibration is an optimization problem: we adjust the free parameters to minimize the squared difference between simulated and experimental voltage, summed over time. This uses the adjoint method (via Jutul.jl) and the LBFGS algorithm under the hood — it runs several simulations internally, so it takes noticeably longer than a single `solve` (expect a few minutes). If it's taking too long during the session, you can loosen the stopping tolerance to finish earlier, at the cost of a slightly less tight fit: `solve(calibration; grad_tol = 1e-4, obj_change_tol = 1e-4)`.

In [128]:
cell_parameters_calibrated, history = solve(calibration)

print_info(calibration)

Let's run a new simulation with the calibrated parameters and compare all three: original simulation, calibrated simulation, and experimental data.

In [129]:
sim_calibrated = Simulation(model_setup, cell_parameters_calibrated, cycling_protocol)
output_calibrated = solve(sim_calibrated);

In [130]:
simdata_time_calibrated = output_calibrated.time_series["Time"]
simdata_voltage_calibrated = output_calibrated.time_series["Voltage"]

f = Figure(size = (700, 400))
ax = Axis(f[1, 1], title = "0.5C discharge: before vs. after calibration", xlabel = "Time / s", ylabel = "Voltage / V")
lines!(ax, simdata_time_original, simdata_voltage_original; linewidth = 4, label = "Simulation: original parameters")
lines!(ax, simdata_time_calibrated, simdata_voltage_calibrated; linewidth = 4, label = "Simulation: calibrated parameters")
scatter!(ax, expdata_05C[:, 1], expdata_05C[:, 2]; markersize = 12, label = "Experimental data")
axislegend(ax, position = :lb)
f

In [131]:
GLMakie.closeall()

This was a "low-rate" calibration against a 0.5C curve, fitting the electrodes' usable stoichiometric range. The natural next step is a "high-rate" calibration against a faster discharge curve, this time freeing the `ReactionRateConstant` and `DiffusionCoefficient` of both electrodes instead — those are the parameters that mainly control rate-dependent behavior. You'll do exactly that in the assignment below.

## Assignment — does your calibration actually generalize?

### Part 1 - high-rate calibration

As mentioned in the calibration section above, the usual next step is a second calibration stage against a faster discharge curve, this time freeing the `ReactionRateConstant` and `DiffusionCoefficient` of both electrodes instead of the stoichiometric coefficients — those are the parameters that mainly control rate-dependent behavior. This repo also includes an experimental **2C** discharge curve for the same cell (`data/xu_2015_voltage_curve_2C.csv`).

Starting from a simulation of `cell_parameters_calibrated` run at 2C, set up a new `VoltageCalibration` against that 2C data (hint: same `free_calibration_parameter!` pattern as before), free the following parameters for both `NegativeElectrode` and `PositiveElectrode` with these bounds, solve it, and store the resulting cell parameters in `cell_parameters_calibrated_highrate`:

- `ReactionRateConstant`: `lower_bound = 1e-16`, `upper_bound = 1e-10`
- `DiffusionCoefficient`: `lower_bound = 1e-16`, `upper_bound = 1e-12`

In [ ]:
expdata_2C = CSV.read("data/xu_2015_voltage_curve_2C.csv", DataFrame);

cycling_protocol_2C = deepcopy(cycling_protocol)
cycling_protocol_2C["DRate"] = 2.0

sim_for_highrate_calibration = Simulation(model_setup, cell_parameters_calibrated, cycling_protocol_2C);

# --- set up a VoltageCalibration against expdata_2C using sim_for_highrate_calibration,
#     free the ReactionRateConstant and DiffusionCoefficient of both electrodes,
#     solve it, and store the resulting cell parameters in `cell_parameters_calibrated_highrate`. ---

calibration_high = VoltageCalibration(expdata_2C[:, 1], expdata_2C[:, 2], sim_for_highrate_calibration);

free_calibration_parameter!(calibration_high,
    ["NegativeElectrode", "ActiveMaterial", "ReactionRateConstant"];
    lower_bound = 1e-16, upper_bound = 1.e-10)
free_calibration_parameter!(calibration_high,
    ["PositiveElectrode", "ActiveMaterial", "ReactionRateConstant"];
    lower_bound = 1e-16, upper_bound = 1e-10)
free_calibration_parameter!(calibration_high,
    ["NegativeElectrode", "ActiveMaterial", "DiffusionCoefficient"];
    lower_bound = 1e-16, upper_bound = 1e-12)
free_calibration_parameter!(calibration_high,
    ["PositiveElectrode", "ActiveMaterial", "DiffusionCoefficient"];
    lower_bound = 1e-16, upper_bound = 1e-12);
# -------------------------------------------------------------------------------

VoltageCalibration([0.0, 110.7248520710059, 246.5976331360946, 394.8224852071005, 543.0473372781064, 691.2721893491125, 839.4970414201184, 963.0177514792899, 1086.5384615384614, 1201.1834319526629, 1308.8757396449705, 1433.1360946745563, 1557.3964497041416, 1667.085798816568, 1765.9023668639052], [3.209675159469554, 3.1512831528809433, 3.1099256158462376, 3.1020113840628363, 3.102733381593856, 3.1003710115125824, 3.0788855622350915, 3.0558404084833604, 3.0093540608782012, 2.991453690880859, 2.986565529396953, 2.9643132317764067, 2.9420609341558603, 2.6549401464601954, 2.003591789416258], Simulation(LithiumIonBattery("Setup object for a P2D lithium-ion model", {
    "ButlerVolmer" => "Standard"
    "RampUp" => "Sinusoidal"
    "Metadata" =>     {
        "Description" => "Default model settings for a P2D simulation including a current ramp up, excluding current collectors and SEI effects."
        "Title" => "p2d"
    }
    "PotentialFlowDiscretization" => "GeneralAD"
    "TransportInSo

In [141]:
cell_parameters_calibrated_highrate, history = solve(calibration);

print_info(calibration)

In [142]:
sim_highrate = Simulation(model_setup, cell_parameters_calibrated_highrate, cycling_protocol_2C)
output_highrate = solve(sim_highrate)

sim_lowrate_at_2C = Simulation(model_setup, cell_parameters_calibrated, cycling_protocol_2C)
output_lowrate_at_2C = solve(sim_lowrate_at_2C)

f = Figure(size = (700, 400))
ax = Axis(f[1, 1], title = "2C discharge: low-rate vs. high-rate calibration", xlabel = "Time / s", ylabel = "Voltage / V")
lines!(ax, output_lowrate_at_2C.time_series["Time"], output_lowrate_at_2C.time_series["Voltage"]; linewidth = 4, label = "Simulation: 0.5C-calibrated parameters")
lines!(ax, output_highrate.time_series["Time"], output_highrate.time_series["Voltage"]; linewidth = 4, label = "Simulation: high-rate calibrated parameters")
scatter!(ax, expdata_2C[:, 1], expdata_2C[:, 2]; markersize = 12, label = "Experimental data (2C)")
axislegend(ax, position = :lb)
f

You calibrated `cell_parameters_calibrated` against a single 0.5C voltage curve and 2C curve. A good calibration should capture real physics, not just trace one curve — so let's put it to the test on data it has never seen.

### Part 2 - test on an unseen rate

This repo also includes an experimental **1C** discharge curve for the same cell (`data/xu_2015_voltage_curve_1C.csv`). Run your calibrated parameters at 1C and compare against that curve.

In [143]:
expdata_1C = CSV.read("data/xu_2015_voltage_curve_1C.csv", DataFrame);

# --- build a 1C cycling protocol (hint: deepcopy `cycling_protocol` and change DRate),
#     run cell_parameters_calibrated through it with model_setup, and solve.
#     Store the result in a variable called `output_test`. ---

cycling_protocol_1C = deepcopy(cycling_protocol)
cycling_protocol_1C["DRate"] = 2.0

sim_1C = Simulation(model_setup, cell_parameters_calibrated_highrate, cycling_protocol_1C);
output_test = solve(sim_1C);

# -------------------------------------------------------------------------------

SimulationOutput({
    "CycleNumber" => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
    "Current" => [0.01903195513426309, 0.14538178464043344, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880633, 0.32236041534880

In [ ]:
f = Figure(size = (700, 400))
ax = Axis(f[1, 1], title = "1C discharge: does the calibration generalize?", xlabel = "Time / s", ylabel = "Voltage / V")
lines!(ax, output_test.time_series["Time"], output_test.time_series["Voltage"]; linewidth = 4, label = "Simulation: calibrated at 0.5C and 2C, run at 1C")
scatter!(ax, expdata_1C[:, 1], expdata_1C[:, 2]; markersize = 12, label = "Experimental data (1C)")
axislegend(ax, position = :lb)
f

In [136]:
GLMakie.closeall()